# What changed

Which of this branch's commits does the loss-of-flow accident need, and what does each one do?

The LOFA benchmark ladder in `benchmarks/lofa/` was run at every commit of the series with one fixed set
of stage definitions. The benchmark itself grew during the first four commits, so commits 1 to 3 were
measured with the benchmark as it stands from commit 4 on. Then each commit after the fourth was
reverted alone on the branch tip and the two hardest stages were run again.

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
ladder = pd.read_csv("results/ladder-by-commit.csv", dtype={"sha": str})
ladder = ladder[~ladder["index"].isin([10, 12])]
ladder["commit"] = ladder["index"].astype(str) + "  " + ladder["sha"] + "  " + ladder["subject"]
order = ladder.drop_duplicates("index").sort_values("index")["commit"]
table = ladder.pivot(index="commit", columns="stage", values="status").reindex(order).fillna("")
table.index.name = None
table.columns.name = None


def shade(status):
    if status == "PASS":
        return "background-color: #d9f2d9; color: #1e5c1e"
    return "background-color: #ececec; color: #666666"


table.style.map(shade).set_properties(**{"text-align": "center"})

,A,B,C,D,E,F,G,H
1 b923864 Fix silent failure modes in the solvers and aggregator core,CRASH,PASS,PASS,TIMEOUT,TIMEOUT,CRASH,CRASH,
"2 6443b32 Fix correctness bugs in components, correlations and property fits",CRASH,PASS,PASS,TIMEOUT,TIMEOUT,CRASH,CRASH,
3 de6a2ff Smooth residual discontinuities and rework event handling,CRASH,PASS,PASS,PASS,PASS,PASS,PASS,
4 e14e18f Globalize the steady-state solve and stop saturated runs loudly,PASS,PASS,PASS,PASS,PASS,PASS,PASS,
5 8ce7266 Make solver failures speak domain terms,PASS,PASS,PASS,PASS,PASS,PASS,PASS,
6 0273f48 Validate systems at construction time,PASS,PASS,PASS,PASS,PASS,PASS,PASS,
"7 007f947 Guard the physics wrappers, quiet healthy runs, add a failure-modes guide",PASS,PASS,PASS,PASS,PASS,PASS,PASS,
8 978e7c8 Correct CHF orientation under reversed flow; adiabatic wall-less channels,PASS,PASS,PASS,PASS,PASS,PASS,PASS,
9 a07ccb1 Compose forced and natural convection monotonically in the HTC,PASS,PASS,PASS,PASS,PASS,PASS,PASS,
11 15e15ec Add a steady-state guess toolkit,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS


Stages, from the benchmark README: A, steady state from a uniform ballpark guess at full power; B, expert
guess at 1 kW; C, choreographed transient at loose tolerance; D, flapper opening time independent of the
output grid; E, tight-tolerance transient; F, single channel at full power with a ramped scram; G, the
general four-channel case; H, the guess toolkit on that case. Two unrelated commits of the original
series (loss-of-coolant transients and the visualization module) were left out of this branch and are
dropped from the table. The branch's toolkit commit, 15e15ec, is a cherry-pick of the series' 66926e7, and
its row is labelled with the branch's id.

The two regression guards pass throughout. At commit 1 the four-channel steady state cannot be solved
from the ballpark guess, the single-channel transient dies in the coastdown at 28 s, and the
tight-tolerance runs do not finish. At commit 2 the four-channel steady state solves and its transient
dies at 82 s. At commit 3 every
transient completes with the final numbers. Commit 4 makes the ballpark steady solve pass at full
single-channel power. Nothing changes afterwards, and the guess-toolkit stage arrives with the toolkit.

In [2]:
loo = pd.read_csv("results/leave-one-out.csv", dtype={"reverted": str})
loo["subject"] = loo["subject"].str.strip()
branch = ["8ce7266", "0273f48", "007f947", "978e7c8", "a07ccb1", "15e15ec"]
loo = loo.pivot(index=["reverted", "subject"], columns="stage", values="status").reset_index()
loo.columns.name = None
loo.sort_values("reverted", key=lambda c: c.map(branch.index), ignore_index=True)

,reverted,subject,F,G
0,8ce7266,Make solver failures speak domain terms,entangled,entangled
1,0273f48,Validate systems at construction time,entangled,entangled
2,007f947,"Guard the physics wrappers, quiet healthy runs...",entangled,entangled
3,978e7c8,Correct CHF orientation under reversed flow; a...,PASS,PASS
4,a07ccb1,Compose forced and natural convection monotoni...,PASS,PASS
5,15e15ec,Add a steady-state guess toolkit,PASS,PASS


Reverting the CHF fix, the heat-transfer commit or the toolkit leaves both stages passing with the same
numbers. The three legibility commits cannot be reverted alone because later code builds on them; that
they are not needed rests on commit 4 passing every stage before they exist.

## Needed

The data show that the first four commits are the minimal starting set for the LOFA; they do not show
that the tip needs each individual fix inside commits 1 and 2.

**b923864, silent failures in the solvers and core.** This is the commit that adds the LOFA benchmark
ladder, and the core fixes the next three are built on (commits 3 and 4 edit nine of the library files
that commits 1 and 2 touch). A failed ODE integration used to come back as a result and now raises; a quasi-static run that
stopped early now returns only the times it actually covered. On this case's path: the finite-difference
Jacobian behind `DAE_jacobian` and `ALG_jacobian` gets an absolute step floor of sqrt(eps), because flows
near zero got derivative entries quantized to round-off and a LOFA takes every channel through zero flow;
`State.merge`, which builds the ballpark and expert guesses, stops aliasing its inputs; the hydraulic
guess behind the expert guess supplies the wall HTCs a `ChannelAndContacts` needs.

**6443b32, component and correlation bugs.** Edge-of-range fixes in property fits, correlations and
components. With them the four-channel steady state solves from the ballpark guess where commit 1 stalls.
The fix that does this on its own is the light-water specific heat clamped at 350 °C, below the pole of its fit near
366 °C: applied alone to commit 1, it reproduces commit 2's result, and none of the commit's other file
changes does. Measured with each commit's own copy of the benchmark, commit 2 showed no change; the
difference is the stage definitions. Elenbaas natural convection taking |Ra| with a floor is reached
only when the stress test switches the heat-transfer law.

**de6a2ff, smoothing and events.** Junction mixing, the upwind inlet temperature, the convection-regime
switch and the flapper law had kinks at mdot = 0, where the implicit integrator ground to a halt, and
boolean root functions latched the flapper opening to whichever output point first saw it.
Once-differentiable smoothing and signed, continuous root functions fix both. This is where the LOFA
physics first runs end to end: every transient completes, with the numbers the branch tip gives.

**e14e18f, globalized steady solve and loud saturation.** `hybr` alone stalls from a far guess;
`solve_steady` now falls back through damped Newton and pseudo-transient continuation with unit-aware
scaling, and the single-channel steady state at full power solves from the ballpark guess. Past bulk
saturation the single-phase channel is meaningless; an opt-in guard now stops the run naming the channel
and cells, which is how the stress sweeps end a case that boils.

## Recommended

The legibility commits change what a user sees when something goes wrong: solver failures name the
component, variable and cell instead of an IDA flag, construction rejects mis-wired systems and warns
about suspicious ones, and correlation wrappers raise instead of returning garbage.

The CHF fix reads subcooling by flow direction, which is LOFA physics although this case does not
evaluate CHF. The heat-transfer commit makes the forced/natural blend monotone, which the stress test
needs because it varies the heat-transfer law. The toolkit builds a steady state from the pump flow
alone. The one-line seed fix, found while building this demonstration, stores lumped pressures as
scalars and silences a NumPy warning on every guess.

## What a mistake looks like now

A cut-down LOFA loop for the examples: one channel, the flapper leg and the pump leg, built from the
same components as `showcase/lofa_case.py`.

In [3]:
from showcase import lofa_case as lc
from stream.calculations import Gravity, HeatExchanger, Inertia, Junction, KirchhoffWDerivatives, Pump, Resistor
from stream.calculations.channel import Channel
from stream.calculations.flapper import Flapper, continuously_differentiable_relaxation
from stream.composition import FlowGraph, check_gravity_mismatch, flow_edge, guess_steady_state
from stream.substances import light_water
from stream.utilities import identity


def show(e):
    print(type(e).__name__)
    print(e)
    for note in getattr(e, "__notes__", []):
        print(note)

In [4]:
def small_loop(return_drop=-lc.LENGTH, wire_flapper=True):
    top, bot = Junction(name="J_top"), Junction(name="J_bot")
    channel = Channel(z_boundaries=np.linspace(0.0, lc.LENGTH, 11), fluid=light_water, pipe=lc._pipe(0.002), name="HotChannel")
    flapper = Flapper(open_at_current=0.15, f=5.0, fluid=light_water, area=1e-3, open_rate=1.0,
                      relaxation=continuously_differentiable_relaxation, name="Flapper")
    pump, flywheel = Pump(pressure=lc.DP0, name="Pump"), Inertia(inertia=3e5, name="Flywheel")
    return_leg = (pump, flywheel, HeatExchanger(outlet=40.0, name="HX"),
                  Gravity(fluid=light_water, disposition=return_drop, name="GravityReturn"))
    return FlowGraph(
        flow_edge((top, bot), channel),
        flow_edge((top, bot), flapper, Gravity(fluid=light_water, disposition=lc.LENGTH, name="GravityFlapper"),
                  Resistor(resistance=10.0, name="ResistFlap")),
        flow_edge((bot, top), *return_leg, ref_mdot_for=(flapper,) if wire_flapper else ()),
        inertial_comps=[flywheel, channel], k_constructor=KirchhoffWDerivatives,
        funcs={flapper: {"t": identity}}, reference_node=(top, 2e5),
    )

The return leg's gravity is given the wrong sign, so the loop does not close at zero flow.

In [5]:
try:
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        check_gravity_mismatch(small_loop(return_drop=+lc.LENGTH).kirchhoff)
    for w in caught:
        print(w.category.__name__, w.message)
except Exception as e:
    show(e)

GravityMismatchError
There are unclosed loops when flows are 0.
The following is a report of those loop components and their aggregate pressure difference (in head = 1.0 [Pascal]):
[([Flapper, GravityFlapper, ResistFlap, Pump, Flywheel, HX, GravityReturn], 13735.919866465016), ([HotChannel, Pump, Flywheel, HX, GravityReturn], 13735.919866465016)]


The pump leg is built without `ref_mdot_for`, so the flapper never receives the flow it watches.
Construction does not catch this one: the system builds, the steady state solves because it does not
need that flow, and the first event evaluation of the transient fails with a TypeError.

In [6]:
fg = small_loop(wire_flapper=False)
agr, K = fg.aggregator, fg.kirchhoff
pump = next(c for c in K.components if isinstance(c, Pump))
try:
    y0 = agr.load(guess_steady_state(agr, K, flows={pump: 0.5}))
    y = agr.solve_steady(y0)
    agr.solve(y, time=np.linspace(0.0, 1.0, 3))
except Exception as e:
    show(e)

TypeError
Flapper.event_margin() missing 1 required keyword-only argument: 'ref_mdot'
raised by Calculation 'Flapper' (Flapper.event_margin)
while evaluating Calculation 'Flapper' (Flapper).event_margin at t=0.0


The full four-channel loop with the hot plate's power left unset, then the one-line guess.

In [7]:
agr, K, refs = lc.build(lc.Params())
del agr.funcs[refs["fuels"]["hot"]]["power"]
try:
    lc.guess(agr, K, refs, "toolkit")
except Exception as e:
    show(e)

MissingPowerError
HotFuel has no 'power' func and no entry in powers; its steady power cannot be read from the system (a kinetics node supplies it at run time). Pass powers={'HotFuel': <Watt>}.


Each message points at the components involved, and the guess error also says what to pass.
Construction rejects some mis-wirings, not all: the missing flow reference got through it and surfaced
only when the transient started.

The full description of every change, with the tests that pin it, is in the commit messages on this
branch.